In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install torchcodec
import torch; torch._dynamo.config.recompile_limit = 64;

In [2]:
from datasets import load_dataset
dataset = load_dataset("SetFit/sst5")

Repo card metadata block was not found. Setting CardData to empty.


In [ ]:
import pandas as pd

df1 = dataset["train"].to_pandas()

df2 = dataset["test"].to_pandas()

df3 = dataset["validation"].to_pandas()

df = pd.concat([df1, df2, df3])
df.head()

,text,label,label_text
0,"a stirring , funny and finally transporting re...",4,very positive
1,apparently reassembled from the cutting-room f...,1,negative
2,they presume their audience wo n't sit still f...,1,negative
3,the entire movie is filled with deja vu moments .,2,neutral
4,this is a visually stunning rumination on love...,3,positive


In [ ]:
def create_balanced_dataset(df):

    # Count the instances of 0
    num_spam = df[df["label"] == 0].shape[0]

    # Randomly sample other instances to match the number of 0 instances
    label1_subset = df[df["label"] == 1].sample(num_spam, random_state=123)
    label2_subset = df[df["label"] == 2].sample(num_spam, random_state=123)
    label3_subset = df[df["label"] == 3].sample(num_spam, random_state=123)
    label4_subset = df[df["label"] == 4].sample(num_spam, random_state=123)

    # Combine other subsets with 0
    balanced_df = pd.concat([label1_subset, label2_subset, label3_subset, label4_subset, df[df["label"] == 0]])

    return balanced_df

balanced_df = create_balanced_dataset(df)
balanced_df = balanced_df.drop("label_text", axis=1)
print(balanced_df["label"].value_counts())

label
1    1510
2    1510
3    1510
4    1510
0    1510
Name: count, dtype: int64


In [36]:
def random_split(df, train_frac, validation_frac):
    # Shuffle the entire DataFrame
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    # Calculate split indices
    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    # Split the DataFrame
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)
# Test size is implied to be 0.2 as the remainder

train_df.to_csv("train.csv", index=None)
validation_df.to_csv("validation.csv", index=None)
test_df.to_csv("test.csv", index=None)

In [ ]:
import torch
major_version, minor_version = torch.cuda.get_device_capability()
print(f"Major: {major_version}, Minor: {minor_version}")
from trl import SFTTrainer
import numpy as np
import os
import pandas as pd
import numpy as np
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments, Trainer
from typing import Tuple
import warnings
from typing import Any, Dict, List, Union
from transformers import DataCollatorForLanguageModeling
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
from unsloth import FastModel

NUM_CLASSES = 5 # number of classes in the dataset

max_seq_length = 2048 # Choose any
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+

model_name = "unsloth/gemma-4-E2B";load_in_4bit = False

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,load_in_4bit = load_in_4bit,
    max_seq_length = max_seq_length,
    dtype = dtype,
)

Major: 7, Minor: 5
==((====))==  Unsloth 2026.9.11: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma4 won't work! Using float32.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [ ]:
from peft import LoftQConfig

model = FastModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

trainable parameters: 31039488


In [13]:
# prompt_template = """Here is a movie review:
# {text}

# Classify the sentiment of this review into one of the following:
# class 0: Very Negative
# class 1: Negative
# class 2: Neutral
# class 3: Positive
# class 4: Very Positive

# ### Response:
# class {label}"""

# def formatting_prompts_func(examples):
#     texts = []
#     for text, label in zip(examples["text"], examples["label"]):
#         formatted = prompt_template.format(text=text, label=label) + tokenizer.eos_token
#         texts.append(formatted)
#     return {"text": texts}

# # # اعمال روی دیتای ترین و ولیدیشن
# # train_dataset = train_df.map(formatting_prompts_func, batched=True)
# # val_dataset   = val_df.map(formatting_prompts_func, batched=True)

# # apply formatting_prompts_func to train_df
# train_df['text'] = formatting_prompts_func(train_df)
# train_dataset = datasets.Dataset.from_pandas(train_df,preserve_index=False)

# validation_df['text'] = formatting_prompts_func(validation_df)
# validation_dataset = datasets.Dataset.from_pandas(validation_df,preserve_index=False)

In [22]:
# import datasets

# prompt_template = """Here is a movie review:
# {text}

# Classify the sentiment of this review into one of the following:
# class 0: Very Negative
# class 1: Negative
# class 2: Neutral
# class 3: Positive
# class 4: Very Positive

# ### Response:
# class {label}"""

# # ۱. فرمت‌بندی مستقیم ستون text در دیتای آموزش
# train_df['text'] = [
#     prompt_template.format(text=t, label=l) + tokenizer.eos_token
#     for t, l in zip(train_df['text'], train_df['label'])
# ]
# train_dataset = datasets.Dataset.from_pandas(train_df, preserve_index=False)

# # ۲. فرمت‌بندی مستقیم ستون text در دیتای اعتبارسنجی
# validation_df['text'] = [
#     prompt_template.format(text=t, label=l) + tokenizer.eos_token
#     for t, l in zip(validation_df['text'], validation_df['label'])
# ]
# validation_dataset = datasets.Dataset.from_pandas(validation_df, preserve_index=False)

In [29]:
# import datasets

# prompt_template = """### Instruction:
# Classify the sentiment of this movie review into one of the categories (0, 1, 2, 3, 4):
# - 0: Very Negative
# - 1: Negative
# - 2: Neutral
# - 3: Positive
# - 4: Very Positive

# Review:
# {text}

# ### Response:
# {label}"""

# train_df['formatted_text'] = [
#     prompt_template.format(text=t.strip(), label=l) + tokenizer.eos_token
#     for t, l in zip(train_df['text'], train_df['label'])
# ]

# validation_df['formatted_text'] = [
#     prompt_template.format(text=t.strip(), label=l) + tokenizer.eos_token
#     for t, l in zip(validation_df['text'], validation_df['label'])
# ]

# train_dataset = datasets.Dataset.from_pandas(train_df[['formatted_text']], preserve_index=False)
# validation_dataset = datasets.Dataset.from_pandas(validation_df[['formatted_text']], preserve_index=False)

In [ ]:
import datasets

prompt_template = """### Instruction:
Classify the sentiment of this movie review into one of the classes (0, 1, 2, 3, 4):
- 0: Very Negative
- 1: Negative
- 2: Neutral
- 3: Positive
- 4: Very Positive

Review:
{text}

### Response:
{label}"""

# apply prompt on texts
train_df['formatted_text'] = [
    prompt_template.format(text=t.strip(), label=int(l)) + tokenizer.eos_token
    for t, l in zip(train_df['text'], train_df['label'])
]

validation_df['formatted_text'] = [
    prompt_template.format(text=t.strip(), label=int(l)) + tokenizer.eos_token
    for t, l in zip(validation_df['text'], validation_df['label'])
]

# create new datasets
train_dataset = datasets.Dataset.from_pandas(train_df[['formatted_text']], preserve_index=False)
validation_dataset = datasets.Dataset.from_pandas(validation_df[['formatted_text']], preserve_index=False)


In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis = -1)
    return {"accuracy": accuracy_score(labels, preds)}

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only




# create trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset,
    dataset_text_field = "formatted_text",
    max_seq_length = 512,
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = 10,
        per_device_eval_batch_size = 2,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
    ),
    compute_metrics = compute_metrics,

)

# apply mask on responses
trainer = train_on_responses_only(
    trainer,
    instruction_part = "### Instruction:\n",
    response_part = "### Response:\n",
)

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["formatted_text"] (num_proc=2):   0%|          | 0/5285 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["formatted_text"] (num_proc=2):   0%|          | 0/755 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/5285 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/755 [00:00<?, ? examples/s]

In [ ]:
# start training
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,285 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Step,Training Loss,Validation Loss
10,0.698914,0.622497
20,0.458554,0.515910
30,0.693247,0.548299
40,0.520326,0.510689
50,0.656309,0.540822
60,0.378732,0.498343


Filter:   0%|          | 0/755 [00:00<?, ? examples/s]

In [53]:
from unsloth import FastLanguageModel

# ۱. فعال‌سازی مجدد حالت تمرین و فعال شدن دوباره‌ی گرادیان‌ها
FastLanguageModel.for_training(model)

# ۲. افزایش سقف استپ‌ها (مثلاً از ۶۰ تا ۱۲۰)
trainer.args.max_steps = 60
trainer.args.eval_steps = 20

# ۳. ادامه‌ی آموزش روی همین مدل
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,285 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 31,039,488 of 5,154,217,504 (0.60% trained)


Step,Training Loss,Validation Loss
20,0.450135,0.562187
40,0.405668,0.503108
60,0.248828,0.503928


Step,Training Loss,Validation Loss


In [54]:
import re
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report, accuracy_score
from unsloth import FastLanguageModel

# ۱. قرار دادن مدل در حالت استنتاج بهینه
FastLanguageModel.for_inference(model)

test_prompt_template = """### Instruction:
Classify the sentiment of this movie review into one of the classes (0, 1, 2, 3, 4):
- 0: Very Negative
- 1: Negative
- 2: Neutral
- 3: Positive
- 4: Very Positive

Review:
{text}

### Response:
"""

predictions = []
ground_truths = []
unmatched_count = 0
sample = test_df[:250]
print(f"Starting evaluation on {len(sample)} test samples...")

for _, row in tqdm(sample.iterrows(), total=len(sample), desc="Testing"):
    review_text = str(row["text"]).strip()
    true_label = int(row["label"])

    prompt = test_prompt_template.format(text=review_text)
    # Fix: Explicitly pass text as a list using the keyword argument
    inputs = tokenizer(text=[prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 2,
            use_cache = True,
            temperature = 0.01  # خروجی قطعی Greedy
        )

    # استخراج توکن‌های تولیدشده بعد از پرامپت
    prompt_len = inputs.input_ids.shape[1]
    pred_text = tokenizer.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()

    # پیدا کردن اولین عدد در بازه 0 تا 4
    match = re.search(r"[0-4]", pred_text)
    if match:
        pred_label = int(match.group(0))
    else:
        # در صورت عدم تولید رقم، کلاسی پیش‌فرض یا برچسب نامعتبر در نظر گرفته می‌شود
        pred_label = -1
        unmatched_count += 1

    predictions.append(pred_label)
    ground_truths.append(true_label)

# ۲. محاسبه و نمایش گزارش عملکرد
acc = accuracy_score(ground_truths, predictions) * 100
print(f"\n==========================================")
print(f"Final Test Accuracy: {acc:.2f}%")
print(f"Unmatched Predictions: {unmatched_count}")
print(f"==========================================\n")

target_names = ["0: Very Neg", "1: Neg", "2: Neutral", "3: Pos", "4: Very Pos"]
print(classification_report(ground_truths, predictions, labels=[0, 1, 2, 3, 4], target_names=target_names))

Starting evaluation on 250 test samples...


Testing: 100%|██████████| 250/250 [01:58<00:00,  2.11it/s]


Final Test Accuracy: 56.40%
Unmatched Predictions: 0

              precision    recall  f1-score   support

 0: Very Neg       0.54      0.78      0.64        50
      1: Neg       0.41      0.43      0.42        54
  2: Neutral       0.59      0.39      0.47        49
      3: Pos       0.61      0.43      0.50        47
 4: Very Pos       0.70      0.80      0.75        50

    accuracy                           0.56       250
   macro avg       0.57      0.56      0.55       250
weighted avg       0.57      0.56      0.55       250



In [55]:
df = pd.DataFrame({'Pred': predictions, 'Ground': ground_truths})
df.head(30)

,Pred,Ground
0,0,1
1,1,2
2,1,0
3,4,3
4,4,4
5,3,3
6,1,1
7,2,2
8,0,1
9,3,4
